# Task 01: Text Preprocessing and Cleaning with NLTK

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/nlp-learning-journey/blob/main/examples/nlp-nltk-tutorials/task-01-text-preprocessing.ipynb)

## Overview

Text preprocessing is the foundation of any NLP pipeline. Raw text from various sources (websites, documents, social media) often contains noise, inconsistencies, and formatting issues that can negatively impact downstream NLP tasks. This tutorial covers essential text cleaning and preprocessing techniques using NLTK.

## What You'll Learn

- Convert text to lowercase
- Remove HTML tags and URLs
- Handle special characters and punctuation
- Remove extra whitespace and newlines
- Handle contractions and abbreviations
- Remove numbers (when appropriate)
- Apply multiple preprocessing steps in a pipeline

## Why Text Preprocessing Matters

Proper text preprocessing:
- Reduces vocabulary size and computational complexity
- Improves model accuracy by normalizing text
- Removes noise that doesn't contribute to meaning
- Creates consistent input for machine learning models
- Helps with feature extraction and analysis

## Environment Setup

In [ ]:
# Environment Detection and Setup
import sys
import subprocess
import os

# Detect the runtime environment
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

print(f"Environment detected:")
print(f"  - Local: {IS_LOCAL}")
print(f"  - Google Colab: {IS_COLAB}")
print(f"  - Kaggle: {IS_KAGGLE}")

# Install required packages
required_packages = ["nltk"]

print("\nInstalling required packages...")
for package in required_packages:
    if IS_COLAB or IS_KAGGLE:
        !pip install -q {package}
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], 
                      capture_output=True)
    print(f"✓ {package}")

## Import Libraries

In [ ]:
import nltk
import re
import string
from html import unescape

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

print("Libraries imported successfully!")

## 1. Case Normalization

Converting text to lowercase ensures that words like "Hello", "hello", and "HELLO" are treated as the same word.

In [ ]:
# Example with English text
english_text = "Hello! My name is JOHN. How are YOU today?"
print(f"Original: {english_text}")
print(f"Lowercase: {english_text.lower()}")

# Example with Vietnamese text
vietnamese_text = "Xin CHÀO! Tên TÔI là John."
print(f"\nOriginal: {vietnamese_text}")
print(f"Lowercase: {vietnamese_text.lower()}")

## 2. Removing HTML Tags and Entities

In [ ]:
def remove_html(text):
    """Remove HTML tags and decode HTML entities."""
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Decode HTML entities
    text = unescape(text)
    return text

# Test with HTML content
html_text = "<p>Hello &amp; welcome to <b>NLP</b>!</p>"
print(f"Original: {html_text}")
print(f"Cleaned: {remove_html(html_text)}")

## 3. Removing URLs and Email Addresses

In [ ]:
def remove_urls_emails(text):
    """Remove URLs and email addresses from text."""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    return text

# Test
text_with_urls = "Visit https://example.com or email me at user@example.com"
print(f"Original: {text_with_urls}")
print(f"Cleaned: {remove_urls_emails(text_with_urls)}")

## 4. Removing Punctuation and Special Characters

In [ ]:
def remove_punctuation(text, keep_apostrophes=True):
    """Remove punctuation from text."""
    if keep_apostrophes:
        # Keep apostrophes for contractions
        text = re.sub(r"[^\w\s']", '', text)
    else:
        text = text.translate(str.maketrans('', '', string.punctuation))
    return text

# Test
text = "Hello! Don't worry, it's going to be great!!!"
print(f"Original: {text}")
print(f"Without punctuation (keep apostrophes): {remove_punctuation(text, keep_apostrophes=True)}")
print(f"Without punctuation (remove all): {remove_punctuation(text, keep_apostrophes=False)}")

## 5. Removing Extra Whitespace

In [ ]:
def normalize_whitespace(text):
    """Remove extra whitespace and normalize line breaks."""
    # Replace multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    # Strip leading and trailing whitespace
    text = text.strip()
    return text

# Test
messy_text = "  Hello    world!\n\n\n  How   are   you?  "
print(f"Original: '{messy_text}'")
print(f"Cleaned: '{normalize_whitespace(messy_text)}'")

## 6. Removing Numbers

In [ ]:
def remove_numbers(text, replace_with=''):
    """Remove numbers from text."""
    return re.sub(r'\d+', replace_with, text)

# Test
text_with_numbers = "I have 3 apples and 25 oranges in 2024."
print(f"Original: {text_with_numbers}")
print(f"Without numbers: {remove_numbers(text_with_numbers)}")
print(f"Numbers replaced with NUM: {remove_numbers(text_with_numbers, replace_with='NUM')}")

## 7. Handling Contractions

In [ ]:
# Common English contractions
CONTRACTIONS = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'll": "he will",
    "he's": "he is",
    "i'd": "i would",
    "i'll": "i will",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it'd": "it would",
    "it'll": "it will",
    "it's": "it is",
    "let's": "let us",
    "shouldn't": "should not",
    "that's": "that is",
    "there's": "there is",
    "they'd": "they would",
    "they'll": "they will",
    "they're": "they are",
    "they've": "they have",
    "wasn't": "was not",
    "we'd": "we would",
    "we'll": "we will",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "won't": "will not",
    "wouldn't": "would not",
    "you'd": "you would",
    "you'll": "you will",
    "you're": "you are",
    "you've": "you have"
}

def expand_contractions(text, contractions_dict=CONTRACTIONS):
    """Expand contractions in text."""
    pattern = re.compile('({})'.format('|'.join(contractions_dict.keys())), 
                        flags=re.IGNORECASE|re.DOTALL)
    
    def replace(match):
        return contractions_dict[match.group(0).lower()]
    
    return pattern.sub(replace, text)

# Test
text_with_contractions = "I can't believe it's already 2024! We're going to learn NLP."
print(f"Original: {text_with_contractions}")
print(f"Expanded: {expand_contractions(text_with_contractions)}")

## 8. Complete Preprocessing Pipeline

Let's combine all the preprocessing steps into a single pipeline.

In [ ]:
def preprocess_text(text, 
                   lowercase=True,
                   remove_html_tags=True,
                   remove_urls=True,
                   remove_punct=True,
                   remove_nums=False,
                   expand_contractions_flag=True,
                   normalize_ws=True):
    """
    Complete text preprocessing pipeline.
    
    Parameters:
    -----------
    text : str
        Input text to preprocess
    lowercase : bool
        Convert to lowercase
    remove_html_tags : bool
        Remove HTML tags and entities
    remove_urls : bool
        Remove URLs and email addresses
    remove_punct : bool
        Remove punctuation
    remove_nums : bool
        Remove numbers
    expand_contractions_flag : bool
        Expand contractions
    normalize_ws : bool
        Normalize whitespace
        
    Returns:
    --------
    str : Preprocessed text
    """
    # Remove HTML
    if remove_html_tags:
        text = remove_html(text)
    
    # Remove URLs and emails
    if remove_urls:
        text = remove_urls_emails(text)
    
    # Expand contractions
    if expand_contractions_flag:
        text = expand_contractions(text)
    
    # Lowercase
    if lowercase:
        text = text.lower()
    
    # Remove numbers
    if remove_nums:
        text = remove_numbers(text)
    
    # Remove punctuation
    if remove_punct:
        text = remove_punctuation(text, keep_apostrophes=False)
    
    # Normalize whitespace
    if normalize_ws:
        text = normalize_whitespace(text)
    
    return text

## 9. Testing the Complete Pipeline

In [ ]:
# Test with messy English text
messy_text = """
<p>Hello! My name is John &amp; I'm a data scientist.</p>
Visit my website: https://example.com or email me@example.com
I've been working for 5 years in NLP!!!   Don't hesitate to reach out.
"""

print("Original Text:")
print(messy_text)
print("\n" + "="*60 + "\n")

cleaned_text = preprocess_text(messy_text)
print("Cleaned Text:")
print(cleaned_text)

In [ ]:
# Test with Vietnamese text
vietnamese_messy = """
<p>Xin chào! Tên tôi là John.</p>
Tôi yêu học máy và xử lý ngôn ngữ tự nhiên!!!
Website: https://example.com
"""

print("Original Vietnamese Text:")
print(vietnamese_messy)
print("\n" + "="*60 + "\n")

cleaned_vietnamese = preprocess_text(vietnamese_messy)
print("Cleaned Vietnamese Text:")
print(cleaned_vietnamese)

## 10. Practical Exercise

Try preprocessing the following text samples with different settings:

In [ ]:
# Exercise 1: Social media post
social_media_text = """
OMG!!! I can't believe I got 1000 followers today! 🎉🎉🎉
Check out my profile: https://twitter.com/example
DM me at example@email.com for collabs!
#NLP #MachineLearning #AI
"""

print("Social Media Post (Original):")
print(social_media_text)
print("\nCleaned:")
print(preprocess_text(social_media_text, remove_nums=True))

In [ ]:
# Exercise 2: News article snippet
news_text = """
<article>
<h1>Breaking News: AI Advances in 2024</h1>
<p>Scientists at MIT haven't stopped pushing boundaries. 
They've developed a new model that's 50% more efficient!</p>
<p>Learn more at: www.news.com/article123</p>
</article>
"""

print("News Article (Original):")
print(news_text)
print("\nCleaned:")
print(preprocess_text(news_text, remove_nums=False))

## Key Takeaways

1. **Context Matters**: Not all preprocessing steps are appropriate for every task
   - Keep numbers for financial analysis or time series
   - Preserve punctuation for sentiment analysis
   - Keep capitalization for named entity recognition

2. **Pipeline Order**: The sequence of preprocessing steps can affect results
   - Expand contractions before removing punctuation
   - Remove HTML before other cleaning steps
   - Normalize whitespace as the final step

3. **Language Considerations**: Some preprocessing steps work better for specific languages
   - Vietnamese has specific diacritical marks to preserve
   - English contractions need special handling

4. **Always Test**: Verify your preprocessing pipeline with sample data before applying to full dataset

## Next Steps

- Move on to **Task 02: Tokenization Techniques**
- Learn about stopword removal and filtering
- Explore stemming and lemmatization

## Additional Resources

- [NLTK Documentation](https://www.nltk.org/)
- [Regular Expressions in Python](https://docs.python.org/3/library/re.html)
- [Text Preprocessing Best Practices](https://developers.google.com/machine-learning/guides/text-classification/step-2)